# Python Standard Library — The Built-in Power Tools

## What is the Standard Library and why use it?
Python ships with a massive collection of modules you never need to install. These cover file I/O, date handling, data structures, pattern matching, serialisation, and much more. Mastering the stdlib means you solve most problems without reaching for third-party packages.

No installation needed — everything here is part of Python itself.

## Table of Contents
1. `os` — operating system interface
2. `sys` — interpreter internals
3. `pathlib` — modern file paths
4. `datetime` — dates, times, and durations
5. `collections` — specialised container types
6. `itertools` — iterator building blocks
7. `functools` — higher-order functions
8. `json` — serialisation
9. `re` — regular expressions recap
10. Common Mistakes
11. Exercises + Solutions
12. Mini-Project: File System Reporter

---
## 1. `os` — Operating System Interface
The `os` module lets you interact with the file system, environment variables, and process information in a cross-platform way.

In [ ]:
import os

# Current working directory
print("CWD:", os.getcwd())

# List directory contents
entries = os.listdir(".")
print("First 5 entries:", entries[:5])

# Build paths safely (handles / vs \ automatically)
config_path = os.path.join("config", "settings.json")
print("Config path:", config_path)

# Path checks
print("Exists:", os.path.exists("."))
print("Is file:", os.path.isfile("."))
print("Is dir:", os.path.isdir("."))

# Create nested directories
os.makedirs("temp/sub/deep", exist_ok=True)
print("Directories created")

# Read environment variable with fallback
home = os.environ.get("HOME", os.environ.get("USERPROFILE", "unknown"))
print("Home directory:", home)

In [ ]:
# Real-world: walk a directory tree
import os

for root, dirs, files in os.walk("."):
    # Skip hidden directories
    dirs[:] = [d for d in dirs if not d.startswith(".")]
    level = root.replace(".", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:  # limit depth for demo
        for f in files[:3]:
            print(f"{indent}  {f}")

---
## 2. `sys` — Interpreter Internals
`sys` exposes data about the Python runtime itself — command-line arguments, the module search path, and ways to exit cleanly.

In [ ]:
import sys

print("Python version:", sys.version)
print("Platform:", sys.platform)
print("Executable:", sys.executable)

# sys.argv — command-line arguments (in notebook argv[0] is the kernel script)
print("argv:", sys.argv)

# sys.path — where Python looks for modules
print("\nFirst 3 paths on sys.path:")
for p in sys.path[:3]:
    print(" ", p)

# Add a custom path at runtime
custom = "/my/project/src"
if custom not in sys.path:
    sys.path.insert(0, custom)

# sys.exit() raises SystemExit — useful in scripts
# sys.exit(1)  # exit with error code

---
## 3. `pathlib` — Modern File Paths
`pathlib.Path` treats file paths as objects rather than strings. It is the modern replacement for most `os.path` usage — cleaner syntax, chainable operations, built-in read/write.

In [ ]:
from pathlib import Path

p = Path(".")

print("Absolute path:", p.resolve())
print("Home directory:", Path.home())

# Build paths with / operator
config = Path.home() / "projects" / "config.json"
print("Config:", config)
print("Name:", config.name)       # config.json
print("Stem:", config.stem)       # config
print("Suffix:", config.suffix)   # .json
print("Parent:", config.parent)   # ~/projects

In [ ]:
from pathlib import Path

# Create directory and write/read files
tmp = Path("temp")
tmp.mkdir(exist_ok=True)

note = tmp / "note.txt"
note.write_text("Hello from pathlib!\nLine two.")
print("Written:", note.read_text())

# Glob — find files matching a pattern
print("\nAll .ipynb files recursively:")
for nb in Path(".").glob("*.ipynb"):
    print(" ", nb.name)

# iterdir — like os.listdir but returns Path objects
print("\nContents of temp/:")
for item in tmp.iterdir():
    kind = "dir" if item.is_dir() else "file"
    print(f"  {item.name} ({kind})")

---
## 4. `datetime` — Dates, Times, and Durations
`datetime` gives you objects for representing points in time (`datetime`), calendar dates (`date`), and durations (`timedelta`). It also handles formatting and parsing.

In [ ]:
from datetime import date, time, datetime, timedelta, timezone

# Current date and time
today = date.today()
now = datetime.now()
print("Today:", today)
print("Now:", now)

# Arithmetic with timedelta
in_30_days = today + timedelta(days=30)
print("30 days from now:", in_30_days)

deadline = datetime(2025, 12, 31, 23, 59, 59)
diff = deadline - now
print(f"Days until deadline: {diff.days}")

In [ ]:
from datetime import datetime, timezone

# Format: datetime → string
now = datetime.now()
formatted = now.strftime("%Y-%m-%d %H:%M:%S")
print("Formatted:", formatted)

# Parse: string → datetime
log_entry = "2024-03-15 14:30:00"
parsed = datetime.strptime(log_entry, "%Y-%m-%d %H:%M:%S")
print("Parsed:", parsed, "| Type:", type(parsed))

# Timezone-aware datetime (UTC)
utc_now = datetime.now(timezone.utc)
print("UTC now:", utc_now)
print("ISO format:", utc_now.isoformat())

---
## 5. `collections` — Specialised Container Types
The `collections` module provides alternatives to plain `dict`, `list`, and `tuple` that handle common patterns with less boilerplate.

In [ ]:
from collections import Counter, defaultdict, OrderedDict, namedtuple, deque

# Counter — count hashable objects
words = "the quick brown fox jumps over the lazy dog the fox".split()
count = Counter(words)
print("Most common:", count.most_common(3))
print("'the' appears:", count["the"], "times")

# Combine counters
more_words = Counter(["the", "fox", "cat"])
combined = count + more_words
print("Combined 'the':", combined["the"])

In [ ]:
from collections import defaultdict, namedtuple, deque

# defaultdict — never raises KeyError for missing keys
groups = defaultdict(list)
data = [("fruits", "apple"), ("vegs", "carrot"), ("fruits", "banana"), ("vegs", "broccoli")]
for category, item in data:
    groups[category].append(item)
print("Groups:", dict(groups))

# namedtuple — lightweight immutable record
Point = namedtuple("Point", ["x", "y", "z"])
p = Point(1.0, 2.5, 3.0)
print(f"Point: x={p.x}, y={p.y}, z={p.z}")
print("As dict:", p._asdict())

# deque — double-ended queue, O(1) append/pop from both ends
q = deque([1, 2, 3], maxlen=5)
q.appendleft(0)
q.append(4)
q.append(5)  # maxlen forces eviction of left element
print("Deque:", q)

---
## 6. `itertools` — Iterator Building Blocks
`itertools` provides memory-efficient tools for working with iterators. These compose well with each other and with generators.

In [ ]:
import itertools

# chain — flatten multiple iterables into one
combined = list(itertools.chain([1, 2], [3, 4], [5]))
print("chain:", combined)

# islice — take the first N items from any iterable
from itertools import islice, count
naturals = count(1)  # infinite: 1, 2, 3, ...
first_10 = list(islice(naturals, 10))
print("First 10 naturals:", first_10)

# combinations and permutations
items = ["A", "B", "C"]
print("Combinations(2):", list(itertools.combinations(items, 2)))
print("Permutations(2):", list(itertools.permutations(items, 2)))

# product — cartesian product (like nested for loops)
sizes = ["S", "M", "L"]
colours = ["red", "blue"]
print("Product:", list(itertools.product(sizes, colours)))

In [ ]:
import itertools

# groupby — group consecutive elements by a key
# IMPORTANT: data must be sorted by key first
transactions = [
    {"date": "2024-01", "amount": 100},
    {"date": "2024-01", "amount": 200},
    {"date": "2024-02", "amount": 150},
    {"date": "2024-02", "amount": 50},
]
transactions.sort(key=lambda x: x["date"])
for month, group in itertools.groupby(transactions, key=lambda x: x["date"]):
    total = sum(t["amount"] for t in group)
    print(f"{month}: ${total}")

# cycle — repeat an iterable indefinitely
colours = itertools.cycle(["red", "green", "blue"])
print("\nCycle sample:", [next(colours) for _ in range(7)])

---
## 7. `functools` — Higher-Order Functions
`functools` helps you work with functions as first-class objects — caching results, creating specialised versions of functions, and combining reducers.

In [ ]:
from functools import reduce, partial, lru_cache, wraps

# reduce — fold a sequence into a single value
product = reduce(lambda acc, x: acc * x, [1, 2, 3, 4, 5])
print("Product 1-5:", product)  # 120

# partial — freeze some arguments of a function
def power(base, exponent):
    return base ** exponent

square = partial(power, exponent=2)
cube   = partial(power, exponent=3)
print("square(5):", square(5))  # 25
print("cube(3):",   cube(3))    # 27

In [ ]:
from functools import lru_cache, wraps
import time

# lru_cache — memoize expensive function calls
@lru_cache(maxsize=128)
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

print("fib(35):", fibonacci(35))
print("Cache info:", fibonacci.cache_info())

# wraps — preserve the wrapped function's metadata in decorators
def timer(func):
    @wraps(func)  # without this, func.__name__ would be 'wrapper'
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed:.4f}s")
        return result
    return wrapper

@timer
def slow_sum(n):
    return sum(range(n))

slow_sum(1_000_000)
print("Function name preserved:", slow_sum.__name__)

---
## 8. `json` — Serialisation
`json` converts Python objects to JSON strings (and files) and back. Essential for APIs, config files, and data exchange.

In [ ]:
import json

data = {
    "name": "Alice",
    "age": 30,
    "scores": [95, 87, 92],
    "active": True,
    "address": None
}

# dict → JSON string
raw = json.dumps(data)
pretty = json.dumps(data, indent=2, sort_keys=True)
print("Raw:", raw)
print("Pretty:\n", pretty)

# JSON string → dict
restored = json.loads(raw)
print("\nRestored type:", type(restored))
print("Name:", restored["name"])

In [ ]:
import json
from pathlib import Path

config_file = Path("temp/config.json")

# Write to file
config = {"debug": True, "host": "localhost", "port": 8080}
with config_file.open("w") as f:
    json.dump(config, f, indent=2)

# Read from file
with config_file.open() as f:
    loaded = json.load(f)
print("Loaded config:", loaded)

# Custom encoder for non-serialisable types
from datetime import datetime

class DateTimeEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, datetime):
            return obj.isoformat()
        return super().default(obj)

event = {"name": "launch", "when": datetime.now()}
print(json.dumps(event, cls=DateTimeEncoder))

---
## 9. `re` — Regular Expressions Recap
Quick reference for the patterns you'll reach for most often day-to-day.

In [ ]:
import re

text = "Contact us at support@example.com or sales@company.org for pricing."

# findall — return all non-overlapping matches as a list
emails = re.findall(r"[\w.+-]+@[\w-]+\.[a-z]{2,}", text)
print("Emails found:", emails)

# search — find first match anywhere in string
m = re.search(r"(\w+)@(\w+)\.([\w.]+)", text)
if m:
    print("Full match:", m.group(0))
    print("Groups:", m.groups())

# sub — replace matches
redacted = re.sub(r"[\w.+-]+@[\w-]+\.[a-z]{2,}", "[REDACTED]", text)
print("Redacted:", redacted)

# compile for reuse
EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+\.[a-z]{2,}", re.IGNORECASE)
print("Compiled match:", bool(EMAIL_RE.search(text)))

---
## 10. Common Mistakes

**`os.path` vs `pathlib`** — mixing them creates messy code. Pick one (prefer `pathlib` for new code).

**`datetime.now()` vs `datetime.utcnow()`** — `utcnow()` returns a naive datetime; use `datetime.now(timezone.utc)` instead to get an aware datetime.

**`groupby` without sorting** — `itertools.groupby` only groups *consecutive* equal elements. Always sort by the key first.

**Mutating the default in `defaultdict`** — `defaultdict(list)` creates a new list per key, which is fine. But `defaultdict(lambda: [])` does the same thing with more typing.

**Forgetting `@wraps`** — decorators that don't use `@wraps(func)` silently replace `__name__`, `__doc__`, and `__module__`, breaking introspection and documentation tools.

**`json.dumps` on non-serialisable types** — `datetime`, `set`, and custom objects raise `TypeError`. Use a custom encoder or convert to strings/lists first.

---
## 11. Exercises

**Exercise 1:** Use `collections.Counter` to count character frequencies in a string, then print the 5 most common characters.

**Exercise 2:** Write a function that accepts a directory path and returns a `dict` mapping file extension (e.g. `.py`) to a list of filenames with that extension. Use `pathlib`.

**Exercise 3:** Use `functools.lru_cache` to memoize a function that computes `n!` (factorial). Verify that repeated calls hit the cache.

In [ ]:
# Solution 1
from collections import Counter

text = "programming with python is powerful and productive"
char_count = Counter(c for c in text if c != " ")
print("Top 5 characters:", char_count.most_common(5))

In [ ]:
# Solution 2
from pathlib import Path
from collections import defaultdict

def group_by_extension(directory: str) -> dict:
    groups = defaultdict(list)
    for item in Path(directory).iterdir():
        if item.is_file():
            groups[item.suffix].append(item.name)
    return dict(groups)

result = group_by_extension(".")
for ext, files in sorted(result.items()):
    print(f"{ext or '(no ext)'}: {files}")

In [ ]:
# Solution 3
from functools import lru_cache

@lru_cache(maxsize=None)
def factorial(n: int) -> int:
    if n <= 1:
        return 1
    return n * factorial(n - 1)

print("10! =", factorial(10))
print("10! =", factorial(10))  # from cache
print(factorial.cache_info())

---
## 12. Mini-Project: File System Reporter
Scan a directory using `pathlib` + `os` + `collections.Counter`, count file types, find the largest files, and format the report with `datetime`.

In [ ]:
from pathlib import Path
from collections import Counter
from datetime import datetime
import os

def file_system_report(directory: str = ".", top_n: int = 5) -> str:
    root = Path(directory).resolve()
    all_files = [f for f in root.rglob("*") if f.is_file()]

    if not all_files:
        return "No files found."

    # Count by extension
    ext_counter = Counter(f.suffix.lower() or "(no ext)" for f in all_files)

    # Find largest files
    files_with_size = [(f, f.stat().st_size) for f in all_files]
    largest = sorted(files_with_size, key=lambda x: x[1], reverse=True)[:top_n]

    # Build report
    lines = [
        "=" * 60,
        f"FILE SYSTEM REPORT",
        f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        f"Directory: {root}",
        "=" * 60,
        f"\nTotal files: {len(all_files)}",
        "\nFile types:",
    ]
    for ext, count in ext_counter.most_common():
        bar = "#" * count
        lines.append(f"  {ext:15s} {count:4d}  {bar}")

    lines.append(f"\nTop {top_n} largest files:")
    for path, size in largest:
        size_kb = size / 1024
        lines.append(f"  {size_kb:8.1f} KB  {path.relative_to(root)}")

    return "\n".join(lines)


report = file_system_report(".")
print(report)

# Also save to file
Path("temp/report.txt").write_text(report)
print("\nReport saved to temp/report.txt")